# 🧭 Tutorial: plotting significant electrodes per condition on the brain

A guided, line-by-line walkthrough of the code behind `plot_subjects.ipynb`,
`dcc_scripts/vis/plot_sig_electrodes_dcc.py`, and
`dcc_scripts/vis/condition_plot_specs.py`.

**How to use this notebook**
- Read each explanation, then **run the code cell under it**. Most cells run on
  small *toy data* defined here, so you can execute everything **without the
  cluster or any real iEEG data**. Cells that would need real data are clearly
  marked `# (conceptual — needs real data)` and are not meant to be run.
- After a concept, there's a **🧠 Quiz** (answer hidden behind a triangle you
  click) and sometimes an **✍️ Assignment** (a small thing for you to code).
- The functions defined here are **copies** of the real ones so the tutorial is
  self-contained. Each is labeled with the file it lives in for real.

Let's go. 🚀

## Part 0 — The big picture

**Goal of the real code:** take groups of *significant electrodes* (from stats
you already ran) for one or more **conditions**, and:
1. draw them on a standard "average" brain (`fsaverage`), **each condition a
   different color**, with electrodes significant in *more than one* condition
   drawn in a single **overlap color** (black); and
2. make a **histogram** of which anatomical **ROIs** (brain regions) those
   electrodes fall in.

**The files involved**

| File | Role |
|------|------|
| `plot_sig_electrodes_dcc.py` | the engine: all the helper functions + `main(args)` |
| `condition_plot_specs.py` | the **registry** of named comparisons (what to plot) |
| `run_plot_sig_electrodes_dcc.py` | picks a comparison + settings, calls `main` |
| `sbatch_/submit_*.sh` | launch it on the DCC cluster |
| `plot_subjects.ipynb` | interactive version of the same thing |

The rest of this tutorial builds up the engine piece by piece, then shows how
the registry and launch scripts sit on top.

**🧠 Quiz**

Why do we plot everyone's electrodes on one shared `fsaverage` brain instead of each subject's own brain?

<details>
<summary>👉 Click to reveal the answer</summary>

Because we want to compare/aggregate electrodes **across subjects** in a common anatomical space. Each subject's electrode coordinates are transformed into the `fsaverage` template so a group of electrodes from many people can be shown on a single brain and compared by location.

</details>

## Part 1 — Setup & paths (`run_*` / notebook top cell)

Every entry-point starts by making sure Python can import the project. The real
code does this:

```python
if os.path.exists("/hpc/home"):                     # on the DCC cluster
    USER = os.environ.get("USER")
    sys.path.append(f"/hpc/home/{USER}/coganlab/{USER}/GlobalLocal/IEEG_Pipelines/")
    current_script_dir = os.path.dirname(os.path.abspath(__file__))
else:                                               # local machine
    sys.path.append("C:/Users/jz421/Desktop/GlobalLocal/IEEG_Pipelines/")
    ...
project_root = os.path.abspath(os.path.join(current_script_dir, "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
```

**Line by line:**
- `os.path.exists("/hpc/home")` is the cheap "am I on the cluster?" test.
- `IEEG_Pipelines/` is added to `sys.path` because the `ieeg` toolbox isn't
  pip-installed into the environment — it's a sibling folder.
- `project_root` is the `GlobalLocal/` root. Adding it to `sys.path` lets
  `import src.analysis...` work no matter where the script is launched from.
- `sys.path.insert(0, ...)` (front of the list) makes the project win over any
  same-named package elsewhere.

**🧠 Quiz**

In a **notebook**, `__file__` isn't defined. What does the real code do about that, and why?

<details>
<summary>👉 Click to reveal the answer</summary>

It wraps `os.path.abspath(__file__)` in a `try/except NameError` and falls back to `os.getcwd()`. In a script `__file__` is the script path; in a notebook there is no `__file__`, so accessing it raises `NameError`, and we use the working directory instead.

</details>

## Part 2 — The data model (this is the important mental model)

Before any plotting, four kinds of objects flow through the code. Let's define
small **toy versions** of each so later cells can actually run.

**(a) `subjects`** — IDs *with* leading zeros, e.g. `"D0057"`. Note the same
person appears *without* zeros (`"D57"`) inside the `ieeg` toolbox. Handling
this mismatch is a recurring theme.

**(b) `sig_chans_per_subject`** — `{subject_with_zeros: [electrode names]}`: the
significant channels for one condition/contrast, loaded from
`sig_chans_{subject}_{root}.json`.

**(c) `subjects_electrodes_to_ROIs_dict`** — per subject, a `default_dict`
mapping each electrode → its anatomical label (Destrieux atlas / aseg). Used for
ROI filtering and the histograms.

**(d) channel names per subject** — every electrode a subject has, in order.
Real code gets this from `subject_to_info(subject)["ch_names"]`. We'll mock it.

In [ ]:
# --- Toy data used throughout the tutorial -------------------------------
# Every electrode each subject has, IN ORDER (mimics subject_to_info(subj)["ch_names"]).
mock_ch_names = {
    "D57": ["RAI1", "RAI2", "RAI3", "LPF1", "LPF2"],   # 5 channels
    "D63": ["RFG1", "RFG2", "LTO1", "LTO2"],           # 4 channels
    "D65": ["RAH1", "RAH2", "RAH3"],                   # 3 channels
}

def subject_to_info(subject):
    """Mock of ieeg.viz.mri.subject_to_info: returns something with ['ch_names']."""
    return {"ch_names": mock_ch_names[subject]}

# Significant channels for one condition (keys have leading zeros).
sig_chans_per_subject = {
    "D0057": ["RAI1", "LPF2"],
    "D0063": ["RFG2"],
    "D0065": [],                  # a subject with no sig electrodes
}

# Electrode -> anatomical ROI, per subject.
subjects_electrodes_to_ROIs_dict = {
    "D0057": {"default_dict": {
        "RAI1": "ctx_rh_G_front_middle", "RAI2": "Right-Cerebral-White-Matter",
        "RAI3": "ctx_rh_S_front_sup", "LPF1": "ctx_lh_G_front_sup",
        "LPF2": "ctx_lh_G_front_middle"}},
    "D0063": {"default_dict": {
        "RFG1": "ctx_rh_G_front_sup", "RFG2": "ctx_rh_G_front_middle",
        "LTO1": "ctx_lh_G_occipital_middle", "LTO2": "Left-Cerebral-White-Matter"}},
    "D0065": {"default_dict": {
        "RAH1": "Right-Amygdala", "RAH2": "ctx_rh_G_front_sup",
        "RAH3": "Right-Cerebral-White-Matter"}},
}

print("Toy data ready. Subjects:", list(mock_ch_names))

**🧠 Quiz**

A "condition" in this code can get its significant electrodes from **two** different sources. What are they?

<details>
<summary>👉 Click to reveal the answer</summary>

1. **`epochs_root_file`** → the `sig_chans_{subject}_{root}.json` files for a single contrast.

2. **`anova_run_dir` + `effect`** → electrodes flagged significant for a *specific effect* (e.g. an interaction like LWPC) inside a within-electrode ANOVA run directory. This is read by `load_significant_electrodes`.

</details>

## Part 3.1 — `strip_leading_zeros` (in `plot_sig_electrodes_dcc.py`)

The single most common chore: `"D0057"` (data/BIDS style) ↔ `"D57"` (`ieeg`
toolbox style), while keeping trailing letters like `"D0107A"` → `"D107A"`.

```python
_SUBJECT_PATTERN = re.compile(r"^D(0*)(\d+)([A-Za-z]*)$")
```
- `^D` — must start with a capital D.
- `(0*)` — group 1: any leading zeros (possibly none).
- `(\d+)` — group 2: the actual number digits.
- `([A-Za-z]*)$` — group 3: optional trailing letters, then end of string.

The function throws away group 1, turns group 2 into an `int` (which drops
leading zeros) and back to a string, and re-attaches group 3.

In [ ]:
import re
_SUBJECT_PATTERN = re.compile(r"^D(0*)(\d+)([A-Za-z]*)$")

def strip_leading_zeros(subject_id):
    match = _SUBJECT_PATTERN.match(subject_id)
    if not match:
        return subject_id                    # not a "D####" id: leave unchanged
    _, numbers, letters = match.groups()
    return f"D{int(numbers)}{letters}"

for s in ["D0057", "D0107A", "D0110", "D5", "fsaverage"]:
    print(f"{s:12} -> {strip_leading_zeros(s)}")

**🧠 Quiz**

Why convert group 2 to `int` and back to `str` instead of using `.lstrip('0')`?

<details>
<summary>👉 Click to reveal the answer</summary>

`int('0057') -> 57 -> '57'` reliably drops leading zeros. `.lstrip('0')` would also work here, but the int round-trip is unambiguous and clearly can't strip a meaningful trailing/interior zero. (Both give `D57`; the int form is just the choice made in the code.)

</details>

**✍️ Assignment**

Write a one-liner that turns a list of with-zeros ids `['D0057', 'D0107A']` into their no-zeros forms using `strip_leading_zeros`. Then check: what does `strip_leading_zeros('X12')` return, and why?

In [ ]:
# ✍️ your turn:
ids = ["D0057", "D0107A"]
# no_zeros = ...
# print(no_zeros)
# print(strip_leading_zeros("X12"))

<details><summary>👉 Solution</summary>

```python
no_zeros = [strip_leading_zeros(s) for s in ids]      # ['D57', 'D107A']
strip_leading_zeros("X12")                            # 'X12'  -> the regex
# requires a leading 'D', so no match -> returned unchanged.
```
</details>

## Part 3.2 — Reshaping helpers: `collapse_rois_to_subject_dict` & `tuples_to_subject_dict`

Different sources hand us electrodes in different shapes; we normalize every
condition to the same `{subject_with_zeros: [names]}` shape.

**`collapse_rois_to_subject_dict`** flattens `{roi: {subject: [elecs]}}`
(what ROI-filtering produces) into `{subject: [elecs]}`, dropping duplicates (an
electrode can sit in two ROIs).

**`tuples_to_subject_dict`** groups `[(subject, electrode), ...]` (what the
ANOVA loader returns) into `{subject: [electrodes]}`.

In [ ]:
def collapse_rois_to_subject_dict(sig_electrodes_per_subject_roi):
    collapsed = {}
    for _roi, subject_dict in sig_electrodes_per_subject_roi.items():
        for subject, electrodes in subject_dict.items():
            bucket = collapsed.setdefault(subject, [])
            bucket.extend(e for e in electrodes if e not in bucket)  # de-dup
    return collapsed

def tuples_to_subject_dict(sub_elec_tuples):
    grouped = {}
    for subject, electrode in sub_elec_tuples:
        bucket = grouped.setdefault(subject, [])
        if electrode not in bucket:
            grouped[subject].append(electrode)
    return grouped

# demo
roi_shaped = {
    "lpfc": {"D0057": ["RAI1", "LPF2"], "D0063": ["RFG2"]},
    "occ":  {"D0057": ["LPF2"], "D0063": ["LTO1"]},        # LPF2 repeats -> de-duped
}
print("collapsed:", collapse_rois_to_subject_dict(roi_shaped))

anova_tuples = [("D0057", "RAI1"), ("D0057", "RAI1"), ("D0057", "LPF2"), ("D0063", "RFG2")]
print("from tuples:", tuples_to_subject_dict(anova_tuples))

**🧠 Quiz**

`setdefault(subject, [])` — what does it do on the *first* vs a *later* visit to the same subject key?

<details>
<summary>👉 Click to reveal the answer</summary>

On the first visit the key doesn't exist, so it inserts `subject: []` and returns that new list. On later visits the key exists, so it just returns the existing list (the default is ignored). Either way you get back the list to append to.

</details>

## Part 3.3 — `get_condition_electrodes`: the two-source dispatcher

This is the function that turns one condition's config `cfg` into
`{subject_with_zeros: [names]}`. Simplified skeleton:

```python
def get_condition_electrodes(subjects, cfg, task, LAB_root, rois_dict=None, ...):
    if cfg.get("anova_run_dir"):                          # --- Source 2: ANOVA
        sub_elec = load_significant_electrodes(
            cfg["anova_run_dir"], roi=cfg.get("anova_roi"),
            effect=cfg.get("effect"), use_fdr=cfg.get("use_fdr", True),
            p_thresh=cfg.get("p_thresh", 0.05))
        return tuples_to_subject_dict(sub_elec)

    sig_chans_per_subject = get_sig_chans_per_subject(                  # Source 1
        subjects, cfg["epochs_root_file"], task=task, LAB_root=LAB_root)
    if not rois_dict:
        return {subj: list(chans) for subj, chans in sig_chans_per_subject.items()}
    # else: filter to ROIs, then collapse -> {subject: [names]}
    ...
```

**Key idea:** the `cfg` dict *is* the switch. If it has an `anova_run_dir` key we
go the ANOVA route; otherwise we go the `sig_chans` route. Both routes end in the
same `{subject: [names]}` shape, so everything downstream is source-agnostic.

**🧠 Quiz**

For an `anova_run_dir` condition, the global `rois_dict` (anatomical ROI filter) is **ignored**. Why is that the right behavior?

<details>
<summary>👉 Click to reveal the answer</summary>

Because a within-electrode ANOVA run is **already ROI-scoped** — you ran it on a chosen set of ROIs. Re-filtering by anatomical labels would be redundant and could silently drop electrodes the ANOVA legitimately found. The `sig_chans` route, by contrast, starts from *all* significant channels, so an optional ROI filter is useful there.

</details>

## Part 3.4 — ⭐ The global index space (`build_global_index_map`, `electrodes_to_global_indices`)

This is the conceptual heart of the plotting. `plot_on_average` doesn't take
electrode *names* — it addresses electrodes by a single **flat integer index**
that runs across **all subjects concatenated in order**.

Imagine stacking every subject's channel list end-to-end:

```
        D57 (5 chans)      D63 (4 chans)     D65 (3 chans)
index:  0  1  2  3  4  |  5  6  7  8  |  9  10 11
        RAI1..... LPF2 |  RFG1... LTO2 | RAH1.. RAH3
```

So D63's `RFG2` (its local index 1) becomes **global index 5 + 1 = 6**. The
"5" is D63's **offset** = the total number of channels before it.

`build_global_index_map` computes each subject's offset; `electrodes_to_global_indices`
uses it to turn names into global indices (skipping electrodes that don't exist,
e.g. dropped in preprocessing).

In [ ]:
from collections import OrderedDict

def build_global_index_map(subjects_no_zeros):
    offsets = OrderedDict()
    running = 0
    for subject in subjects_no_zeros:
        info = subject_to_info(subject)
        offsets[subject] = (running, info["ch_names"])
        running += len(info["ch_names"])          # next subject starts after this one
    return offsets

def electrodes_to_global_indices(condition_electrodes, offsets):
    indices = []
    for subject_with_zeros, electrodes in condition_electrodes.items():
        subject = strip_leading_zeros(subject_with_zeros)   # 'D0057' -> 'D57'
        if subject not in offsets:
            continue
        offset, ch_names = offsets[subject]
        for electrode in electrodes:
            if electrode in ch_names:
                indices.append(offset + ch_names.index(electrode))
            else:
                print(f"  (skipping {electrode}: not in {subject})")
    return sorted(set(indices))

subjects_no_zeros = ["D57", "D63", "D65"]
offsets = build_global_index_map(subjects_no_zeros)
for subj, (off, names) in offsets.items():
    print(f"{subj}: offset={off}, {len(names)} channels")

condA = {"D0057": ["RAI1", "LPF2"], "D0063": ["RFG2"]}
print("condA global indices:", electrodes_to_global_indices(condA, offsets))

**🧠 Quiz**

Using the toy data, what global index does D65's `RAH2` get? Work it out by hand, then check with code.

<details>
<summary>👉 Click to reveal the answer</summary>

D65's offset is 5 + 4 = **9**. `RAH2` is local index **1** in `['RAH1','RAH2','RAH3']`. So global index = 9 + 1 = **10**.

</details>

**✍️ Assignment**

Confirm your answer: build a condition dict `{'D0065': ['RAH2']}` and run it through `electrodes_to_global_indices`. Then add a **bogus** electrode `'RAH9'` and observe what happens.

In [ ]:
# ✍️ your turn:
# print(electrodes_to_global_indices({"D0065": ["RAH2"]}, offsets))
# print(electrodes_to_global_indices({"D0065": ["RAH2", "RAH9"]}, offsets))

<details><summary>👉 Solution</summary>

```python
electrodes_to_global_indices({"D0065": ["RAH2"]}, offsets)          # [10]
electrodes_to_global_indices({"D0065": ["RAH2", "RAH9"]}, offsets)  # prints
# "(skipping RAH9: not in D65)" and returns [10]. Missing electrodes are skipped,
# not errored — important when preprocessing dropped some channels.
```
</details>

> **Why one shared index map for all conditions?** Because if every condition is
> expressed in the *same* global index space, an electrode significant in two
> conditions gets the **same integer** in both — so we can find overlaps with a
> plain set intersection. That's exactly what the next function does.

## Part 3.5 — ⭐ `split_unique_and_overlap`: the coloring logic

Given each condition's set of global indices, we want: electrodes unique to one
condition keep that condition's color; electrodes in **≥2** conditions are drawn
once in the overlap color (black).

```python
def split_unique_and_overlap(condition_indices):
    counts = Counter()
    for idx_set in condition_indices.values():
        counts.update(idx_set)                      # how many conditions each index is in
    overlap = sorted(idx for idx, n in counts.items() if n > 1)
    overlap_set = set(overlap)
    unique = OrderedDict(
        (name, sorted(i for i in idx_set if i not in overlap_set))
        for name, idx_set in condition_indices.items())
    return unique, overlap
```

In [ ]:
from collections import Counter

def split_unique_and_overlap(condition_indices):
    counts = Counter()
    for idx_set in condition_indices.values():
        counts.update(idx_set)
    overlap = sorted(idx for idx, n in counts.items() if n > 1)
    overlap_set = set(overlap)
    unique = OrderedDict(
        (name, sorted(i for i in idx_set if i not in overlap_set))
        for name, idx_set in condition_indices.items())
    return unique, overlap

# condA and condB share exactly one electrode (D57 LPF2 == global index 4)
condA_idx = set(electrodes_to_global_indices({"D0057": ["RAI1", "LPF2"], "D0063": ["RFG2"]}, offsets))
condB_idx = set(electrodes_to_global_indices({"D0057": ["LPF2"], "D0065": ["RAH2"]}, offsets))
condition_indices = OrderedDict([("condA", condA_idx), ("condB", condB_idx)])
print("condA:", sorted(condA_idx), " condB:", sorted(condB_idx))

unique, overlap = split_unique_and_overlap(condition_indices)
print("unique:", dict(unique))
print("overlap:", overlap)

**🧠 Quiz**

If you passed **three** conditions and an electrode was significant in exactly 2 of them, which group would it land in — and what color would it get?

<details>
<summary>👉 Click to reveal the answer</summary>

It lands in **overlap** (the rule is `n > 1`, i.e. two *or more*). It would be drawn in the single overlap color (black). The code doesn't distinguish "in 2 vs in 3" — any sharing counts as overlap.

</details>

**✍️ Assignment**

Modify a *copy* of `split_unique_and_overlap` called `split_unique_pairs_and_all` that returns three things: unique, `shared_by_exactly_two`, and `shared_by_all`. (Hint: compare `n` to `len(condition_indices)`.)

<details><summary>👉 Solution</summary>

```python
def split_unique_pairs_and_all(condition_indices):
    n_cond = len(condition_indices)
    counts = Counter()
    for idx_set in condition_indices.values():
        counts.update(idx_set)
    shared_by_all = sorted(i for i, n in counts.items() if n == n_cond)
    shared_two   = sorted(i for i, n in counts.items() if 1 < n < n_cond)
    all_shared   = set(shared_by_all) | set(shared_two)
    unique = OrderedDict(
        (name, sorted(i for i in s if i not in all_shared))
        for name, s in condition_indices.items())
    return unique, shared_two, shared_by_all
```
</details>

## Part 3.6 — ROI histograms (`electrode_roi_counts`, `plot_roi_histogram`)

For each condition's electrodes, count what anatomical region each falls in
(via `default_dict`), optionally dropping white-matter/unknown, then draw a
horizontal bar chart. This cell actually renders a chart. 📊

In [ ]:
import matplotlib.pyplot as plt

def electrode_roi_counts(condition_electrodes, roi_dict, drop_white_matter=True):
    counts = Counter()
    for subject_with_zeros, electrodes in condition_electrodes.items():
        default_dict = roi_dict.get(subject_with_zeros, {}).get("default_dict", {})
        for electrode in electrodes:
            roi = default_dict.get(electrode, "Unknown")
            if drop_white_matter and ("White-Matter" in roi or roi in ("Unknown", "unknown")):
                continue
            counts[roi] += 1
    return counts

def plot_roi_histogram(counts, title, color=(0.2, 0.4, 0.8), top_n=None):
    if not counts:
        print("nothing to plot"); return
    items = counts.most_common(top_n)
    labels = [l for l, _ in items][::-1]      # largest on top
    values = [v for _, v in items][::-1]
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(labels) + 1.5))
    ax.barh(labels, values, color=color, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Number of significant electrodes"); ax.set_title(title)
    for y, v in enumerate(values):
        ax.text(v, y, f" {v}", va="center")
    plt.tight_layout(); plt.show()

# Use a bigger toy condition so the histogram has a few bars.
big_condition = {"D0057": ["RAI1", "RAI2", "RAI3", "LPF2"], "D0063": ["RFG1", "RFG2"]}
counts = electrode_roi_counts(big_condition, subjects_electrodes_to_ROIs_dict)
print("ROI counts:", dict(counts))
plot_roi_histogram(counts, title="Toy condition — ROI histogram", color=(1.0, 0.0, 0.0))

**🧠 Quiz**

In the toy data, `RAI2` maps to `Right-Cerebral-White-Matter`. With `drop_white_matter=True`, does it appear in the histogram? What if you pass `False`?

<details>
<summary>👉 Click to reveal the answer</summary>

With `True` it is **skipped** (the `"White-Matter" in roi` check catches it). With `False` it would be counted under the `Right-Cerebral-White-Matter` bar.

</details>

## Part 4 — Drawing on the brain (`plot_on_average`)  *(conceptual — needs real data)*

The real drawing calls (from `main`) look like this — you can't run them here
(no `mne`/brain data), but the logic is simple:

```python
fig = None
for name, indices in plot_groups.items():          # each condition
    fig = plot_on_average(subjects_no_zeros, picks=sorted(indices),
                          color=conditions[name]["color"], fig=fig, ...)
if overlap:
    fig = plot_on_average(subjects_no_zeros, picks=overlap,
                          color=overlap_color, fig=fig, ...)
fig.save_image(".../brain_<label>.png")
```

**Three things to notice:**
1. `picks=sorted(indices)` — the global indices from Part 3.4, **sorted**.
   `plot_on_average` walks subjects in order and subtracts each subject's channel
   count as it goes, so the picks **must be sorted ascending** to line up.
2. `fig=fig` — the first call makes a new `Brain`; every later call **draws onto
   the same brain**, which is how multiple colors end up on one image.
3. `fig.save_image(path)` writes the PNG. On the headless cluster this needs
   off-screen rendering (`PYVISTA_OFF_SCREEN=true` + `xvfb-run`), which the
   module and sbatch script set up for you.

**🧠 Quiz**

Why must `picks` be passed **sorted ascending**? (Think about how `plot_on_average` peels off one subject at a time.)

<details>
<summary>👉 Click to reveal the answer</summary>

Internally it iterates subjects in order and, after handling each subject, subtracts that subject's channel count from the remaining picks — assuming the picks belonging to the current subject are the *smallest* remaining ones. If the list weren't sorted, indices would be attributed to the wrong subject. Sorting guarantees each subject's picks come first in the remaining list.

</details>

## Part 5 — The modular registry (`condition_plot_specs.py`)

This file has **no heavy dependencies**, so we can import the **real thing** and
poke at it. 🎉 It holds:
- a **color palette** + `assign_colors` (auto-color conditions that don't specify one),
- **builders** `sig_chans(...)` and `anova_effect(...)` (one-liner condition specs),
- `anova_run(condition_label, n_subjects)` (rebuilds an ANOVA run path),
- the `PLOT_CONDITION_SETS` **registry** of named comparisons,
- `resolve_plot_set(label)` (looks up a set + fills in colors/defaults).

In [ ]:
import os, sys
# make sure the project root is importable (adjust if your cwd differs)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dcc_scripts.vis.condition_plot_specs import (
    DEFAULT_PALETTE, assign_colors, sig_chans, anova_effect, anova_run,
    resolve_plot_set, PLOT_CONDITION_SETS,
)

print("Available comparisons:", sorted(PLOT_CONDITION_SETS))
print("\nPalette (first 3):", DEFAULT_PALETTE[:3])

### 5a — `anova_run` rebuilds the path from three pieces

```python
def anova_run(condition_label, n_subjects):
    return os.path.join(POWER_FIGS_BASE, ANOVA_EPOCHS_ROOT,
                        f"anova_within_{ANOVA_UNIT}",
                        f"{condition_label}_{n_subjects}_subjects")
```

So `anova_run("stimulus_lwpc_conditions", 24)` →
`.../<ANOVA_EPOCHS_ROOT>/anova_within_electrode/stimulus_lwpc_conditions_24_subjects`.
The three pieces come from env vars (with defaults) so you can point at different
runs without editing the file.

In [ ]:
# Override the location pieces via env vars, then rebuild (import already done,
# so set os.environ BEFORE import in real use; here we just show the path shape).
print(anova_run("stimulus_lwpc_conditions", 24))
print(anova_run("stimulus_switch_type_by_incongruent_proportion_conditions", 24))

### 5b — `resolve_plot_set` gives you a ready-to-plot spec

In [ ]:
spec = resolve_plot_set("lwpc_vs_lwps")
for name, cfg in spec["conditions"].items():
    print(f"{name:6} color={cfg['color']}  effect={cfg.get('effect')}")
print("mutually_exclusive:", spec["mutually_exclusive"], "| overlap_color:", spec["overlap_color"])

**🧠 Quiz**

In `lwpc_vs_lwps`, neither condition specifies a `color`. Where do red and blue come from?

<details>
<summary>👉 Click to reveal the answer</summary>

`resolve_plot_set` runs the conditions through `assign_colors`, which fills any missing `color` from `DEFAULT_PALETTE` in order — so the 1st condition gets `DEFAULT_PALETTE[0]` (red) and the 2nd gets `DEFAULT_PALETTE[1]` (blue).

</details>

**✍️ Assignment**

Build an **ad-hoc** 3-condition set inline (no registry edit): LWPC interaction, LWPS interaction, and the congruency **main** effect from the `stimulus_experiment_conditions` run — all with 24 subjects. Use `anova_effect` + `anova_run`, wrap in `assign_colors`, and print each condition's color.

In [ ]:
from collections import OrderedDict
# ✍️ your turn — fill in the three conditions:
# my_set = assign_colors(OrderedDict([
#     ("lwpc", anova_effect(anova_run(..., 24), "C(congruency):C(incongruentProportion)")),
#     ("lwps", ...),
#     ("congruency", ...),
# ]))
# for n, c in my_set.items():
#     print(n, c["color"], c["effect"])

<details><summary>👉 Solution</summary>

```python
my_set = assign_colors(OrderedDict([
    ("lwpc",       anova_effect(anova_run("stimulus_lwpc_conditions", 24),
                                "C(congruency):C(incongruentProportion)")),
    ("lwps",       anova_effect(anova_run("stimulus_lwps_conditions", 24),
                                "C(switchType):C(switchProportion)")),
    ("congruency", anova_effect(anova_run("stimulus_experiment_conditions", 24),
                                "C(congruency)")),
]))
for n, c in my_set.items():
    print(n, c["color"], c["effect"])
# -> lwpc red, lwps blue, congruency green (DEFAULT_PALETTE[0..2]).
```
To actually plot it, drop `my_set` into `conditions` in the notebook's config
cell (or add it to `PLOT_CONDITION_SETS` under a new label).
</details>

## Part 6 — How `main(args)` ties it together, and the launch chain

`main(args)` is just the pieces above in order:
1. Build the electrode→ROI map (for filtering + histograms).
2. For each condition → `get_condition_electrodes` → `{subject: [names]}`; also
   dump `sig_electrodes_<condition>.json` for the record.
3. `build_global_index_map` (shared) → per-condition global index **sets**.
4. `split_unique_and_overlap` (if `mutually_exclusive`).
5. Combined brain (overlay all + overlap), then one brain per condition.
6. ROI histogram per condition.

**The launch chain on the cluster:**

```
submit_plot_sig_electrodes_dcc.sh     # pick label(s), export env vars, sbatch
   └─ sbatch_plot_sig_electrodes_dcc.sh   # SLURM + conda + xvfb-run
        └─ run_plot_sig_electrodes_dcc.py # reads PLOT_SET_LABEL, builds args
             └─ main(args)                # the engine (this whole tutorial)
```

To plot a **different** comparison you change one thing: the `PLOT_SET_LABEL`
(or the `PLOT_SETS=(...)` list in the submit script). Everything else flows from
the registry.

**🧠 Quiz**

You change `PLOT_SET_LABEL` from `lwpc_vs_lwps` to `congruency_vs_switchType` and resubmit. Which files change, and which don't?

<details>
<summary>👉 Click to reveal the answer</summary>

**Nothing in the Python changes.** Only the env var passed by the submit script differs. `run_*` calls `resolve_plot_set(<new label>)`, which returns a different `conditions` dict from the registry, and `main` does the rest. The engine, builders, and helpers are all comparison-agnostic.

</details>

## 🏁 Capstone assignment

Put it all together **on the toy data** (no cluster needed):

1. Define **three** toy conditions as `{subject_with_zeros: [names]}` such that:
   - one electrode is shared by all three,
   - one pair shares an extra electrode,
   - each condition has at least one unique electrode.
2. Convert each to global index sets with `electrodes_to_global_indices`.
3. Run `split_unique_and_overlap` and print unique-per-condition + overlap.
4. For each condition, print its `electrode_roi_counts`.
5. **Predict** how many *distinct colors* would appear on the brain (including
   the overlap color), then verify against your output.

Write it in the cell below; a reference solution is hidden underneath.

In [ ]:
# ✍️ capstone — your turn:

<details><summary>👉 Reference solution</summary>

```python
condsA = {"D0057": ["RAI1", "LPF2"], "D0063": ["RFG2"]}          # RAI1 unique
condsB = {"D0057": ["LPF2"], "D0063": ["RFG2"], "D0065": ["RAH2"]}  # RAH2 unique
condsC = {"D0057": ["LPF2"], "D0063": ["RFG1", "RFG2"]}          # RFG1 unique

ci = OrderedDict([
    ("A", set(electrodes_to_global_indices(condsA, offsets))),
    ("B", set(electrodes_to_global_indices(condsB, offsets))),
    ("C", set(electrodes_to_global_indices(condsC, offsets))),
])
unique, overlap = split_unique_and_overlap(ci)
print("unique:", dict(unique))     # each has its own unique index
print("overlap:", overlap)         # LPF2 (idx 4, in all 3) + RFG2 (idx 6, in all 3)

for name, conds in [("A", condsA), ("B", condsB), ("C", condsC)]:
    print(name, dict(electrode_roi_counts(conds, subjects_electrodes_to_ROIs_dict)))

# Distinct colors on the brain = 3 condition colors + 1 overlap color = 4,
# BUT only if each condition still has ≥1 unique electrode after removing overlap.
```
Note: in this construction `LPF2` (idx 4) and `RFG2` (idx 6) are both in all
three → overlap; `RAI1`, `RAH2`, `RFG1` stay unique. So you'd see red/blue/green
uniques + black overlap = **4 colors**.
</details>

## Where to go next

- Open the real files side-by-side and match each function to its tutorial
  section:
  - `dcc_scripts/vis/plot_sig_electrodes_dcc.py`
  - `dcc_scripts/vis/condition_plot_specs.py`
  - `dcc_scripts/vis/run_plot_sig_electrodes_dcc.py`
- Then run the interactive `plot_subjects.ipynb` on real data (needs the `ieeg`
  env + Box/cluster access) and watch the same functions produce actual brains.

You now understand the whole pipeline end to end. 🧠✨